### Verificações de qualidade e regras de negócio

As verificações abaixo avaliam limites, integridade e relações entre os serviços. Valores ausentes mantidos por falta de informação não são tratados como inconsistências automaticamente.

In [3]:
# Leitura do CSV
import pandas as pd
df_silver = pd.read_csv('base_silver.csv', encoding="utf-8", sep=",")

In [4]:
# Valores fora dos intervalos esperados e categorias válidas.
regras_validacao = {
    "DuracaoContrato fora de 0 a 72": int((df_silver["DuracaoContrato"].notna() & ~df_silver["DuracaoContrato"].between(0, 72)).sum()),
    "CobrancaMensal negativa": int((df_silver["CobrancaMensal"] < 0).sum()),
    "CobrancaTotal negativa": int((df_silver["CobrancaTotal"].dropna() < 0).sum()),
    "RendaMensal menor ou igual a zero": int((df_silver["RendaMensal"].dropna() <= 0).sum()),
    "Senioridade fora de 0/1": int((df_silver["Senioridade"].notna() & ~df_silver["Senioridade"].isin([0, 1])).sum()),
    "Cancelamento diferente de Yes/No": int((~df_silver["Cancelamento"].isin(["Yes", "No"])).sum()),
    "ID de cliente duplicado": int(df_silver["id_cliente"].duplicated().sum()),
    "Linhas totalmente duplicadas": int(df_silver.duplicated().sum()),
}

# Coerência entre PhoneService e MultipleLines.
mascara_sem_telefone = df_silver["ServicoTelefonico"].eq("No")
regras_validacao["Sem telefone mas MultipleLines diferente de 'No phone service'"] = int(
    (mascara_sem_telefone & df_silver["MultiplasLinhas"].ne("No phone service")).sum()
)

# Coerência dos seis serviços adicionais de internet.
colunas_sem_internet = ["SegurancaOnline", "BackupOnline", "ProtecaoDispositivo", "SuporteTecnico", "StreamingTV", "StreamingFilmes"]
mascara_sem_internet = df_silver["ServicoInternet"].eq("No")
for coluna in colunas_sem_internet:
    regras_validacao[f"Sem internet mas {coluna} diferente de 'No internet service'"] = int(
        (mascara_sem_internet & df_silver[coluna].ne("No internet service")).sum()
    )

validacao_regras = pd.DataFrame.from_dict(regras_validacao, orient="index", columns=["quantidade_violacoes"])
display(validacao_regras)

,quantidade_violacoes
DuracaoContrato fora de 0 a 72,0
CobrancaMensal negativa,0
CobrancaTotal negativa,0
RendaMensal menor ou igual a zero,0
Senioridade fora de 0/1,0
Cancelamento diferente de Yes/No,0
ID de cliente duplicado,0
Linhas totalmente duplicadas,0
Sem telefone mas MultipleLines diferente de 'No phone service',0
Sem internet mas SegurancaOnline diferente de 'No internet service',0
